In [1]:
!pip install ultralytics -q

import os
import yaml
import json
import torch
import pandas as pd
from ultralytics import YOLO

# ==========================================
# 1. HARDWARE CHECK & KAGGLE SETUP
# ==========================================
num_gpus = torch.cuda.device_count()
gpu_devices = [i for i in range(num_gpus)] if num_gpus > 1 else 0

print("="*70)
print(f"HARDWARE: Using {num_gpus}x GPUs (Device IDs: {gpu_devices})")
print("="*70)

# ==========================================
# 2. DATASET PREPARATION (Symlink & Convert)
# ==========================================
BASE_DIR = '/kaggle/input/datasets/prosenjitmondol/complete-vindr-spinexr'
IMG_TRAIN = '/kaggle/input/datasets/prosenjitmondol/complete-vindr-spinexr/vindr-spinexr-a-large-annotated-medical-image-dataset/vindr-spinexr-a-large-annotated-medical-image-dataset/train_png'
IMG_VAL = f'{BASE_DIR}/vindr-spinexr-a-large-annotated-medical-image-dataset/vindr-spinexr-a-large-annotated-medical-image-dataset/test_png'
COCO_TRAIN = f'{BASE_DIR}/coco format/train_coco.json'
COCO_VAL = f'{BASE_DIR}/coco format/test_coco.json'

WORK_BASE = '/kaggle/working/vindr_yolo'
os.makedirs(f'{WORK_BASE}/images/train', exist_ok=True)
os.makedirs(f'{WORK_BASE}/images/val', exist_ok=True)
os.makedirs(f'{WORK_BASE}/labels/train', exist_ok=True)
os.makedirs(f'{WORK_BASE}/labels/val', exist_ok=True)

print("Setting up YOLO dataset and linking images...")

# Symlink images to save space and time
for img in os.listdir(IMG_TRAIN):
    if img.endswith(('.png', '.jpg')):
        dst = os.path.join(f'{WORK_BASE}/images/train', img)
        if not os.path.exists(dst): os.symlink(os.path.join(IMG_TRAIN, img), dst)

for img in os.listdir(IMG_VAL):
    if img.endswith(('.png', '.jpg')):
        dst = os.path.join(f'{WORK_BASE}/images/val', img)
        if not os.path.exists(dst): os.symlink(os.path.join(IMG_VAL, img), dst)

def convert_coco_to_yolo(json_path, labels_dir):
    with open(json_path) as f:
        data = json.load(f)
    cat_map = {cat['id']: i for i, cat in enumerate(data['categories'])}
    img_map = {img['id']: img for img in data['images']}

    for ann in data['annotations']:
        img = img_map[ann['image_id']]
        x_min, y_min, w, h = ann['bbox']
        x_center = (x_min + w / 2) / img['width']
        y_center = (y_min + h / 2) / img['height']
        w_norm = w / img['width']
        h_norm = h / img['height']
        cls_id = cat_map[ann['category_id']]

        txt_filename = img['file_name'].rsplit('.', 1)[0] + '.txt'
        with open(os.path.join(labels_dir, txt_filename), 'a') as out_f:
            out_f.write(f"{cls_id} {x_center} {y_center} {w_norm} {h_norm}\n")
    return {i: cat['name'] for i, cat in enumerate(data['categories'])}

# Convert labels if the folder is empty
if len(os.listdir(f'{WORK_BASE}/labels/train')) == 0:
    print("Converting COCO JSON to YOLO format...")
    class_names_dict = convert_coco_to_yolo(COCO_TRAIN, f'{WORK_BASE}/labels/train')
    convert_coco_to_yolo(COCO_VAL, f'{WORK_BASE}/labels/val')
else:
    print("Labels already converted!")
    with open(COCO_TRAIN) as f:
        data = json.load(f)
        class_names_dict = {i: cat['name'] for i, cat in enumerate(data['categories'])}

# ==========================================
# 3. GENERATE YOLO YAML
# ==========================================
yaml_content = {
    'path': WORK_BASE,
    'train': 'images/train',
    'val': 'images/val',
    'nc': len(class_names_dict),
    'names': class_names_dict
}
YAML_PATH = '/kaggle/working/vindr_data.yaml'
with open(YAML_PATH, 'w') as f:
    yaml.dump(yaml_content, f, sort_keys=False)

print(f"✓ Dataset setup complete! YAML created at {YAML_PATH}")

# ==========================================
# 4. YOLO12 TRAINING CONFIGURATION
# ==========================================
EPOCHS = 50
BATCH_SIZE = 24  # Optimized for T4x2 (12 images per GPU)
IMG_SIZE = 640

train_args = {
    'data': YAML_PATH,
    'epochs': EPOCHS,
    'batch': BATCH_SIZE,
    'imgsz': IMG_SIZE,
    'device': gpu_devices, # Triggers Multi-GPU processing
    
    # Fail-Safe Checkpointing
    'save_period': 1,      
    'save': True,          
    
    # Optimized Hyperparameters matching your paper
    'optimizer': 'AdamW',
    'lr0': 0.0001,
    'lrf': 0.01,
    'momentum': 0.937,
    'weight_decay': 0.0005,
    'box': 7.5,
    'cls': 0.5,
    'dfl': 1.5,
    'copy_paste': 0.2, 
    'mosaic': 1.0,
    'amp': True,
    
    # Project paths specifically for YOLO12
    'project': '/kaggle/working/yolo12_runs',
    'name': 'vindr_yolo12l',
    'seed': 42
}

print("\nDownloading and initializing YOLO12-L...")
model = YOLO('yolo12l.pt') # Loading YOLO12-Large

print(f"\nStarting YOLO12 training for {EPOCHS} epochs...")
print("Weights will automatically save to /kaggle/working/yolo12_runs/vindr_yolo12l/weights/ every epoch.")
results = model.train(**train_args)

# ==========================================
# 5. PAPER-READY OUTPUT FORMATTER
# ==========================================
print("\n" + "="*80)
print("EXTRACTING FINAL METRICS FOR MICCAI TABLE (YOLO12)")
print("="*80)

# Run a final strict validation to get per-class AP50
val_results = model.val(data=YAML_PATH, split='val', iou=0.5, conf=0.25)

# Map YOLO class indices to Paper LT Codes
target_columns = ["LT2(*)", "LT4", "LT6", "LT8", "LT10", "LT11", "LT13", "mAP@0.5"]
yolo_to_lt_map = {
    4: "LT2(*)",  # Disc space narrowing
    3: "LT4",     # Foraminal stenosis
    0: "LT6",     # Osteophytes
    2: "LT8",     # Spondylolysthesis
    1: "LT10",    # Surgical implant
    5: "LT11",    # Vertebral collapse
    6: "LT13"     # Other lesions
}

# Extract mAP50 per class (converted to percentage)
class_indices = val_results.results_dict['classes']
map50_per_class = val_results.box.map50

paper_metrics = {}
for i, cls_idx in enumerate(class_indices):
    lt_code = yolo_to_lt_map[cls_idx]
    paper_metrics[lt_code] = map50_per_class[i] * 100

# Add overall mAP@0.5
overall_map50 = val_results.box.map50.mean() * 100
paper_metrics["mAP@0.5"] = overall_map50

# Print strictly formatted table string
header_str = f"{'Method':<16} " + " ".join([f"{col:<7}" for col in target_columns])
values_str = f"{'Ours (YOLO12-L)':<16} " + " ".join([f"{paper_metrics.get(col, 0.0):<7.2f}" for col in target_columns])

print("\n" + header_str)
print(values_str)
print("="*80)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 98.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23

KeyError: 'classes'